<a href="https://colab.research.google.com/github/scardenol/Stochastic_Optimization_2026/blob/main/Algoritmos/clase3_LShaped_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Método L-Shaped para problemas de dos etapas

Implementación siguiendo la notación de las notas de clase (*Optimización Estocástica — Clase 3*).

**Problema original**

\begin{align*}
&\min_x \; c^T x + \mathcal{Q}(x), \\
&\text{s.a} \quad Ax=b,\; x\ge 0
\end{align*}

Donde $\mathcal{Q}(x) = \mathbb{E}_\xi[Q(x,\xi)]$ es la función de costo esperado de recurso.

**Subproblema de recurso para el escenario $k$** (con probabilidad $p_k$)

$$Q(x,\xi_k) = \min_{y_k\ge 0} \; q_k^T y_k \quad \text{s.a.} \quad W y_k = h_k - T_k x$$

Donde $q_k(\xi_k), h_k(\xi_k), T_k(\xi_k)$.

**La idea del algoritmo:** en vez de resolver el "monolito" (Equivalente Determinista) completo, se descompone en un **Problema Maestro** (decide $x$) y $K$ **subproblemas** (uno por escenario), que le devuelven al Maestro *cortes* (restricciones lineales) hasta converger.

## Cargar paquetes

In [1]:
!pip install gurobipy  # instalar gurobipy, si no está instalado
import numpy as np
import gurobipy as gp
from gurobipy import GRB

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 53.8 MB/s eta 0:00:00


## 1. Estructura de datos del problema

- Se crea una clase `TwoStageProblem` donde se guardan los datos de primera etapa ($c, A, b$) y una lista de escenarios, cada uno con $(p_k, q_k, W, T_k, h_k)$.

- Se asume **recurso fijo**: la matriz $W$ es la misma para todos los escenarios.

In [2]:
class TwoStageProblem:
    """Contenedor simple con los datos del problema de dos etapas."""

    # Constructor que recibe los hiperparámetros de la clase
    def __init__(self, c, A, b, x_ub=None):
        self.c = np.atleast_1d(np.asarray(c, dtype=float)) # vector de costos
        self.A = np.atleast_2d(np.asarray(A, dtype=float)) # matriz de coef.
        self.b = np.atleast_1d(np.asarray(b, dtype=float)) # vector de recursos
        self.n1 = self.c.shape[0]           # número de variables x
        self.x_ub = x_ub                    # cota superior opcional para x
        self.scenarios = []                 # lista de dicts: p, q, W, T, h

    # Función para agregar un escenario
    def add_scenario(self, p, q, W, T, h):
        """Agrega un escenario k con su probabilidad p y datos (q, W, T, h)."""
        self.scenarios.append({
            "p": p,
            "q": np.atleast_1d(np.asarray(q, dtype=float)),
            "W": np.atleast_2d(np.asarray(W, dtype=float)),
            "T": np.atleast_2d(np.asarray(T, dtype=float)),
            "h": np.atleast_1d(np.asarray(h, dtype=float)),
        })

## 2. Algoritmo L-Shaped (single-cut)

Básicamente sigue estos 4 pasos:

- **Paso 0, Inicialización:** $\nu=1$, $\theta$ muy negativo, sin cortes.
- **Paso 1, Problema Maestro:** resolver $\min\{c^Tx+\theta \mid Ax=b,\ \text{cortes actuales},\ x\ge0\}$ y obtener $(x^\nu,\theta^\nu)$.
- **Paso 2, Subproblemas:** para cada escenario $k$, resolver el problema de recurso con $x=x^\nu$.
  - Si es **infactible** → resolver el subproblema de factibilidad (Fase I) para obtener el certificado $\sigma_k^\nu$ y agregar el **corte de factibilidad** $(\sigma_k^\nu)^TT_k\,x \ge (\sigma_k^\nu)^Th_k$.
  - Si es **factible** → guardar el valor óptimo $Q(x^\nu,\xi_k)$ y el dual $\pi_k^\nu$.
  - Si se agregó algún corte de factibilidad, volver al Paso 1.
- **Paso 3, Parada y corte de optimalidad:** calcular $Q(x^\nu)=\sum_k p_k Q(x^\nu,\xi_k)$.
  - Si $\theta^\nu \ge Q(x^\nu)$: **parar**, $x^\nu$ es óptimo.
  - Si no: agregar el **corte de optimalidad** $\theta \ge e^{\nu+1} - E^{\nu+1}x$, donde
  
  \begin{gather*}
  e^{\nu+1}=\sum_k p_k(\pi_k^\nu)^Th_k, \qquad E^{\nu+1}=\sum_k p_k(\pi_k^\nu)^TT_k
  \end{gather*}

In [3]:
class LShapedSolver:
    """Implementa el algoritmo L-Shaped de un solo corte (single-cut)."""

    # Constructor con los hiperparámetros del algoritmo
    def __init__(self, problem, theta_lb=-1e7, tol=1e-6, max_iter=100, verbose=True):
        self.problem = problem
        self.theta_lb = theta_lb
        self.tol = tol
        self.max_iter = max_iter
        self.verbose = verbose

        self.optimality_cuts = []   # lista de (E, e)
        self.feasibility_cuts = []  # lista de (D, d)

    # ---------- Paso 1: Problema Maestro ----------
    def _solve_master(self):
        p = self.problem
        m = gp.Model("maestro")
        m.Params.OutputFlag = 0

        ub = p.x_ub if p.x_ub is not None else GRB.INFINITY
        x = m.addMVar(p.n1, lb=0, ub=ub, name="x")
        theta = m.addVar(lb=self.theta_lb, name="theta")

        m.addConstr(p.A @ x == p.b, name="Ax_b")

        for E, e in self.optimality_cuts:
            m.addConstr(theta >= e - E @ x)
        for D, d in self.feasibility_cuts:
            m.addConstr(D @ x >= d)

        m.setObjective(p.c @ x + theta, GRB.MINIMIZE)
        m.optimize()

        return x.X, theta.X

    # ---------- Paso 2: Subproblema de recurso (primal) ----------
    def _solve_subproblem(self, k, x_val):
        s = self.problem.scenarios[k]
        rhs = s["h"] - s["T"] @ x_val  # h_k - T_k x^nu

        m = gp.Model(f"subproblema_{k}")
        m.Params.OutputFlag = 0

        n2 = s["W"].shape[1]
        y = m.addMVar(n2, lb=0, name="y")
        con = m.addConstr(s["W"] @ y == rhs, name="Wy")
        m.setObjective(s["q"] @ y, GRB.MINIMIZE)
        m.optimize()

        if m.Status == GRB.OPTIMAL:
            return True, m.ObjVal, np.array(con.Pi)  # factible: valor y dual pi_k
        return False, None, None # no factible

    # ---------- Paso 2 (caso infactible): Subproblema de factibilidad (Fase I) ----------
    def _solve_feasibility_subproblem(self, k, x_val):
        s = self.problem.scenarios[k]
        rhs = s["h"] - s["T"] @ x_val

        m = gp.Model(f"factibilidad_{k}")
        m.Params.OutputFlag = 0

        n2 = s["W"].shape[1]
        y = m.addMVar(n2, lb=0, name="y")
        v_plus = m.addMVar(len(rhs), lb=0, name="v_plus")
        v_minus = m.addMVar(len(rhs), lb=0, name="v_minus")

        con = m.addConstr(s["W"] @ y + v_plus - v_minus == rhs, name="Wy_v")
        m.setObjective(v_plus.sum() + v_minus.sum(), GRB.MINIMIZE)
        m.optimize()

        sigma = np.array(con.Pi)  # certificado de infactibilidad sigma_k^nu
        return m.ObjVal, sigma

    # ---------- Ciclo principal ----------
    def solve(self):
        nu = 1 # inicializar la iteración

        while nu <= self.max_iter:
            # Paso 1: resolver el Problema Maestro
            x_val, theta_val = self._solve_master()

            # Paso 2: resolver subproblemas para cada escenario
            feasibility_cut_added = False
            duals, values, probs = [], [], []

            for k, s in enumerate(self.problem.scenarios):
                feasible, obj_val, pi_k = self._solve_subproblem(k, x_val)

                if not feasible: # si NO es factible, agregar corte de factibilidad
                    # Corte de factibilidad: (sigma_k^T T_k) x >= sigma_k^T h_k
                    _, sigma_k = self._solve_feasibility_subproblem(k, x_val)
                    D = sigma_k @ s["T"]
                    d = sigma_k @ s["h"]
                    self.feasibility_cuts.append((D, d))
                    feasibility_cut_added = True
                else: # si es factible, guardar valor y dual
                    duals.append(pi_k)
                    values.append(obj_val)
                    probs.append(s["p"])

            if feasibility_cut_added:
                if self.verbose:
                    print(f"Iter {nu}: x = {x_val} -> infactible, se agrega corte de factibilidad")
                nu += 1
                continue

            # Paso 3: criterio de parada y corte de optimalidad
            Q_x = sum(p * v for p, v in zip(probs, values))  # costo real esperado

            if self.verbose:
                print(f"Iter {nu}: x = {x_val}, theta = {theta_val:.6f}, Q(x) = {Q_x:.6f}")

            if theta_val >= Q_x - self.tol:
                if self.verbose:
                    print(f"\nConvergencia en la iteración {nu}.")
                return {"x": x_val, "theta": theta_val,
                        "obj": self.problem.c @ x_val + theta_val, "iterations": nu}

            # e = sum_k p_k pi_k^T h_k ,   E = sum_k p_k pi_k^T T_k
            e = sum(p * (pi @ s["h"]) for p, pi, s in zip(probs, duals, self.problem.scenarios))
            E = sum(p * (pi @ s["T"]) for p, pi, s in zip(probs, duals, self.problem.scenarios))
            self.optimality_cuts.append((E, e))

            nu += 1

        raise RuntimeError("Se alcanzó max_iter sin convergencia.")

## 3. Validación con ejemplo numérico de clase

$$\min_x Q(x) = \mathbb{E}[\,|\xi - x|\,] \quad \text{s.a.} \quad 0 \le x \le 10$$

con recurso $\min\{y_1+y_2 \mid y_1-y_2=\xi-x,\ y_1,y_2\ge0\} = |\xi-x|$, y $\xi\in\{1,2,4\}$ con probabilidad $1/3$ cada uno.

Al resolver esto a mano llegamos al óptimo $x^*=2$ con un costo esperado mínimo $=1$. Por lo que podemos validar con el código si llegamos al mismo resultado.

In [4]:
# c=0 (no hay costo de primera etapa); usamos A=[[0]], b=[0] (restricción trivial)
# y x_ub=10 para modelar 0 <= x <= 10.
problem = TwoStageProblem(c=[0.0], A=[[0.0]], b=[0.0], x_ub=10.0)

W = [[1.0, -1.0]]   # y1 - y2 = xi - x
for xi in [1.0, 2.0, 4.0]:
    problem.add_scenario(
        p=1 / 3,
        q=[1.0, 1.0],     # min y1 + y2
        W=W,
        T=[[1.0]],        # T_k x = x
        h=[xi],           # h_k = xi
    )

solver = LShapedSolver(problem, theta_lb=-1e4, verbose=True)
result = solver.solve()

print("\nx* =", result["x"], " costo esperado óptimo =", result["obj"])

Restricted license - for non-production use only - expires 2027-11-29
Iter 1: x = [0.], theta = -10000.000000, Q(x) = 2.333333
Iter 2: x = [10.], theta = -7.666667, Q(x) = 7.666667
Iter 3: x = [2.33333333], theta = 0.000000, Q(x) = 1.111111
Iter 4: x = [1.5], theta = 0.833333, Q(x) = 1.166667
Iter 5: x = [2.], theta = 1.000000, Q(x) = 1.000000

Convergencia en la iteración 5.

x* = [2.]  costo esperado óptimo = 1.0


Comparando la salida con la tabla de las notas de clase:
- iteración 1 da $x=0,\ \theta=-\infty,\ Q=7/3\approx2.333$;
- iteración 2 da $x=10,\ \theta=-23/3\approx-7.667$;
- iteración 3 da $x=7/3\approx2.333,\ \theta=0$;
- iteración 4 da $x=1.5,\ \theta\approx0.833$;
- iteración 5 da $x=2,\ \theta=1=Q(2)$ → converge.

Coincide exactamente, por lo que se logra validar el método.

## 4. Ejemplo con corte de factibilidad

Para ver la otra rama del algoritmo, modificamos el recurso para que **solo pueda cubrir faltantes** (no excesos): $y\ge0$ con $y=\xi-x$. Si $x>\xi_k$ para algún escenario, el subproblema es infactible y debe dispararse un **corte de factibilidad** que restrinja a $x\le\min_k \xi_k=1$.

In [5]:
problem2 = TwoStageProblem(c=[1.0], A=[[0.0]], b=[0.0], x_ub=10.0)
W2 = [[1.0]]
for xi in [1.0, 2.0, 4.0]:
    problem2.add_scenario(p=1 / 3, q=[5.0], W=W2, T=[[1.0]], h=[xi])

solver2 = LShapedSolver(problem2, theta_lb=-1e4, verbose=True)
result2 = solver2.solve()

print("\nx* =", result2["x"], " obj* =", result2["obj"])

Iter 1: x = [0.], theta = -10000.000000, Q(x) = 11.666667
Iter 2: x = [10.] -> infactible, se agrega corte de factibilidad
Iter 3: x = [1.], theta = 6.666667, Q(x) = 6.666667

Convergencia en la iteración 3.

x* = [1.]  obj* = 7.66666666666606


En la iteración 2, el Maestro propone $x=10$, que es infactible para los tres escenarios ($y\ge0$ no puede cubrir $\xi-x<0$). Se genera un corte de factibilidad y el Maestro se ve forzado a $x=1=\min_k\xi_k$, que sí es factible para todos los escenarios.

## 5. Actividad: Localización y Capacidad de Centros de Datos

**Contexto**

Una empresa de tecnología planea expandir su infraestructura cloud en una nueva región. Se han identificado
2 ubicaciones potenciales para nuevos centros de datos (CD), que deben abastecer a 3 zonas de clientes
principales.

**Decisiones de 1ra etapa** — capacidad $x_i$ (en Petaflops) a instalar en cada ubicación $i\in\{1,2\}$,
con costo de instalación $f_1=2000$ y $f_2=2200$ (miles de \$ por Petaflop).

**Incertidumbre** — la demanda de cómputo de cada zona de cliente es incierta. Se modelan $K=4$ escenarios
equiprobables ($p_k=0.25$) para el próximo año.

**Decisiones de 2da etapa** — una vez conocida la demanda $d_{jk}$, se asigna el tráfico de cómputo
$y_{ij,k}$ desde cada CD $i$ a cada zona de cliente $j$, con costo operativo agregado $c_{ij}$. Si la
capacidad instalada no basta, se alquila capacidad externa a un costo de penalización $p_{\text{penal}}=3000$
(miles de \$) por Petaflop faltante $s_{j,k}$.

**Objetivo:** minimizar el costo de inversión más el costo esperado de operación y penalizaciones.

---

**Formulación del subproblema de recurso (escenario $k$)**

$$Q(x,\xi_k) = \min_{y_k,s_k} \;\sum_{i=1}^{2}\sum_{j=1}^{3} c_{ij}\,y_{ij,k} \;+\; \sum_{j=1}^{3} p_{\text{penal}}\,s_{j,k}$$

$$\text{s.a.} \quad \sum_{i=1}^{2} y_{ij,k} + s_{j,k} \ge d_{jk}, \quad \forall j \in\{1,2,3\} \quad \text{(balance de demanda)}$$

$$\sum_{j=1}^{3} y_{ij,k} \le x_i, \quad \forall i \in\{1,2\} \quad \text{(límite de capacidad)}$$

$$y_{ij,k}\ge 0,\quad s_{j,k}\ge 0$$

**Problema maestro (1ra etapa)**

$$\min_{x_1,x_2\ge 0} \;\sum_{i=1}^{2} f_i x_i \;+\; \sum_{k=1}^{4} p_k\, Q(x,\xi_k)$$

**Datos**

| | $f_i$ (inversión) |
|---|---|
| CD 1 | 2000 |
| CD 2 | 2200 |

| $c_{ij}$ | Cliente 1 | Cliente 2 | Cliente 3 |
|---|---|---|---|
| CD 1 | 50 | 80 | 100 |
| CD 2 | 90 | 60 | 70 |

| Escenario | $d_1$ | $d_2$ | $d_3$ |
|---|---|---|---|
| $k=1$ (Baja) | 10 | 15 | 12 |
| $k=2$ (Media-Baja) | 20 | 25 | 22 |
| $k=3$ (Media-Alta) | 30 | 40 | 35 |
| $k=4$ (Alta) | 50 | 55 | 48 |

$p_{\text{penal}} = 3000$ (miles de \$ por Petaflop faltante), $p_k = 0.25 \;\forall k$.

**Mapeo a la formulación general del recurso fijo**

Para usar `TwoStageProblem`/`LShapedSolver`, cada restricción de desigualdad se convierte a igualdad
agregando variables de holgura, de modo que $Wy_k = h_k - T_kx$:

- Balance de demanda: $\sum_i y_{ij,k} + s_{j,k} - \text{demslack}_{j,k} = d_{jk}$
- Límite de capacidad: $\sum_j y_{ij,k} + \text{capslack}_{i,k} = x_i \;\Longrightarrow\; T_{i,i}=-1$

In [6]:
# ---------- Datos del problema ----------
c_cost = np.array([2000.0, 2200.0])   # inversión por Petaflop (f_i)
p_pen = 3000.0                        # costo de penalización por Petaflop faltante

demand_scenarios = [
    [10, 15, 12],   # k=1 Baja
    [20, 25, 22],   # k=2 Media-Baja
    [30, 40, 35],   # k=3 Media-Alta
    [50, 55, 48],   # k=4 Alta
]

# q_k: costos operativos c_ij + penalización + 0 para las variables de holgura
q_k = np.array([50, 80, 100,  90, 60, 70,   # y_ij  (CD1->C1,C2,C3, CD2->C1,C2,C3)
                 p_pen, p_pen, p_pen,        # s_j   (faltante por cliente)
                 0, 0,                       # capslack_i (holgura de capacidad)
                 0, 0, 0])                   # demslack_j (holgura de demanda)

# W (5 x 14): filas 0-2 = balance de demanda, filas 3-4 = límite de capacidad
W = np.zeros((5, 14))
W[0, [0, 3, 6, 11]] = [1, 1, 1, -1]   # demanda cliente 1
W[1, [1, 4, 7, 12]] = [1, 1, 1, -1]   # demanda cliente 2
W[2, [2, 5, 8, 13]] = [1, 1, 1, -1]   # demanda cliente 3
W[3, [0, 1, 2, 9]]  = [1, 1, 1, 1]    # capacidad CD 1
W[4, [3, 4, 5, 10]] = [1, 1, 1, 1]    # capacidad CD 2

# T (5 x 2): solo las filas de capacidad dependen de x
T = np.zeros((5, 2))
T[3, 0] = -1.0
T[4, 1] = -1.0

# ---------- Construir el problema con la API de la clase ----------
# No hay restricciones reales de primera etapa (A x = b), así que usamos
# una fila trivial 0*x1 + 0*x2 = 0 para que TwoStageProblem tenga un A, b válidos.
problem_dc = TwoStageProblem(c=c_cost, A=[[0.0, 0.0]], b=[0.0], x_ub=None)

for d in demand_scenarios:
    h_k = np.array([d[0], d[1], d[2], 0.0, 0.0])
    problem_dc.add_scenario(p=0.25, q=q_k, W=W, T=T, h=h_k)

# ---------- Resolver ----------
solver_dc = LShapedSolver(problem_dc, theta_lb=-1e7, verbose=True)
result_dc = solver_dc.solve()

print("\nx* (capacidad CD1, CD2) =", result_dc["x"])
print("Costo total esperado óptimo =", result_dc["obj"])

Iter 1: x = [0. 0.], theta = -10000000.000000, Q(x) = 271500.000000
Iter 2: x = [3481.86440678    0.        ], theta = -10000000.000000, Q(x) = 7000.000000
Iter 3: x = [89.66101695  0.        ], theta = 7000.000000, Q(x) = 64118.220339
Iter 4: x = [51.4548495  0.       ], theta = 119708.193980, Q(x) = 131071.153846
Iter 5: x = [67.02054795  0.        ], theta = 97060.102740, Q(x) = 97075.000000
Iter 6: x = [67.  0.], theta = 97105.000000, Q(x) = 97105.000000

Convergencia en la iteración 6.

x* (capacidad CD1, CD2) = [67.  0.]
Costo total esperado óptimo = 231105.0
